# Treino YOLO26 - Deteccao de EPI (Colab) - v7 (modelo maior + dominio de video)

Roda **sozinho**: prepara os dados, treina, e no final compara v6 x v7 no conjunto de teste real (frames de video que nunca entram no treino), salvando tudo no Drive.

**Por que uma v7** (ver `results/analise_erros/`): no teste real, a maioria das luvas perdidas **nao e pequena** (luvas pretas, de couro de soldador, nitrilicas coloridas, ate 280 px) e as botas perdidas sao escuras, borradas pelo movimento e meio escondidas. Isso e diferenca de dominio (fotos nitidas de obra x frames de video industrial), e nenhuma estrategia so de inferencia (recorte por pessoa, fatias, TTA, 1280 px) melhorou de forma significativa. Entao a v7 ataca o modelo e os dados:

- **Arquitetura YOLO26** (jan/2026, Ultralytics): atribuicao de rotulos que favorece objetos pequenos (STAL) e mais rapida na CPU que a YOLO11. Tamanho **s** por padrao (~3x a capacidade da n usada ate a v6). Comeca dos pesos COCO (nao da para herdar pesos da YOLO11n).
- **Augmentations de video** (Albumentations): borrao de movimento, compressao JPEG, reducao de resolucao, ruido, brilho/contraste - simulam frame de video no treino.
- **Reamostragem**: imagens com luva ou bota entram 2x por epoca (lista `train.txt`, sem copiar arquivo).
- **Frames novos anotados** (opcional): se existir `novos_frames_v7.zip` no Drive, os frames entram no treino (8x, como os da v3).

**O que precisa estar no Drive** (`Meu Drive/TCC/`) - os 4 primeiros ja estao la desde a v6:
```
TCC/
  yolo_dataset/
    yolo_v2_limpo.zip
    novos_frames_dataset.zip
    construction_ppe_dataset.zip
    novos_v5.zip
    novos_frames_v7.zip      <- opcional (frames novos, export YOLO do Roboflow)
    teste_real_v7.zip        <- opcional (teste real novo, export YOLO do Roboflow)
  modelos/
    best_v6.pt               <- models/best_v6.pt do PC (so para a comparacao final)
```
Depois: `Ambiente de execucao > Alterar o tipo` -> **GPU T4** -> `Executar tudo`. Aba aberta, PC sem hibernar.

**Tempo estimado (T4):** YOLO26s em 960 ~18-20 min/epoca -> 50 epocas ~ 15 h, em varias sessoes. Se a cota estiver curta, troque `MODELO_BASE` para `yolo26n.pt` (~8 min/epoca).

**Se a sessao cair**: `Executar tudo` de novo (na MESMA conta) - o treino continua de onde parou; se ja terminou, pula para a avaliacao.

**Resultados** em `Meu Drive/TCC/runs/train-colab-v7/`: `weights/best.pt`, `comparacao_teste_real.csv/.txt`, `results.csv`. No PC: copie o `best.pt` para `models/best_v7.pt` e rode `scripts/avaliar_teste_real.py` (mesmo protocolo de todas as versoes, com intervalo de confianca).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/TCC'
ZIPS = f'{DRIVE}/yolo_dataset'
RUNS = f'{DRIVE}/runs'
RUN_NAME = 'train-colab-v7'
RUN_DIR = f'{RUNS}/{RUN_NAME}'
V6_MODEL = f'{DRIVE}/modelos/best_v6.pt'
DATA_DIR = '/content/data'
CLASSES = ['helmet', 'vest', 'gloves', 'boots', 'goggles']

# ---- configuracao do treino ----
MODELO_BASE = 'yolo26s.pt'   # 'yolo26n.pt' se a cota de GPU estiver curta
IMGSZ = 960
EPOCHS = 50
PATIENCE = 12
REPETE_LUVA_BOTA = 2         # imagens com luva ou bota entram 2x por epoca
REPETE_FRAMES_NOVOS = 8      # frames novos de video (como os da v3)

for p in [f'{ZIPS}/yolo_v2_limpo.zip', f'{ZIPS}/novos_frames_dataset.zip',
          f'{ZIPS}/construction_ppe_dataset.zip', f'{ZIPS}/novos_v5.zip']:
    assert os.path.exists(p), f'Nao encontrado: {p}'
TEM_FRAMES_NOVOS = os.path.exists(f'{ZIPS}/novos_frames_v7.zip')
TEM_TESTE_NOVO = os.path.exists(f'{ZIPS}/teste_real_v7.zip')
print('OK - arquivos no Drive | frames novos:', TEM_FRAMES_NOVOS, '| teste real novo:', TEM_TESTE_NOVO)

## Dados: mesmos da v6 + frames novos (opcional)

In [ ]:
import glob, shutil, yaml

for split in ['train', 'val', 'test']:
    for tipo in ['images', 'labels']:
        os.makedirs(f'{DATA_DIR}/{split}/{tipo}', exist_ok=True)


def copiar(src_root, pares, prefixo=''):
    # pares: [(split de origem, split de destino)]
    for src, dst in pares:
        for tipo in ['images', 'labels']:
            origem = f'{src_root}/{src}/{tipo}'
            for fname in os.listdir(origem):
                shutil.copy(os.path.join(origem, fname), f'{DATA_DIR}/{dst}/{tipo}/{prefixo}{fname}')


!unzip -q -o "{ZIPS}/yolo_v2_limpo.zip" -d /content/v2
copiar('/content/v2/yolo_v2_limpo', [('train', 'train'), ('val', 'val'), ('test', 'test')])

# frames reais do video: train_extra (ja com oversample 8x) entra no treino;
# real_world_test fica separado e nunca e treinado
!unzip -q -o "{ZIPS}/novos_frames_dataset.zip" -d /content/novos
copiar('/content/novos/novos_frames_dataset', [('train_extra', 'train')])
REAL_TEST_DIR = '/content/real_world_test'
shutil.copytree('/content/novos/novos_frames_dataset/real_world_test', REAL_TEST_DIR, dirs_exist_ok=True)

!unzip -q -o "{ZIPS}/construction_ppe_dataset.zip" -d /content/cppe
copiar('/content/cppe/construction_ppe_raw', [('train', 'train'), ('val', 'val'), ('test', 'test')])

!unzip -q -o "{ZIPS}/novos_v5.zip" -d /content/nv5
copiar('/content/nv5/novos_v5', [('train', 'train'), ('valid', 'val'), ('test', 'test')])


def remapear_roboflow(pasta_export, destino_imgs, destino_lbls, prefixo):
    # o Roboflow exporta as classes em ordem alfabetica: remapeia pelo NOME
    nomes = yaml.safe_load(open(glob.glob(f'{pasta_export}/**/data.yaml', recursive=True)[0]))['names']
    n = 0
    for img in glob.glob(f'{pasta_export}/**/images/*', recursive=True):
        base = os.path.splitext(os.path.basename(img))[0]
        lbl = img.replace(f'{os.sep}images{os.sep}', f'{os.sep}labels{os.sep}').rsplit('.', 1)[0] + '.txt'
        linhas = []
        if os.path.exists(lbl):
            for l in open(lbl):
                p = l.split()
                if len(p) >= 5 and nomes[int(p[0])] in CLASSES:
                    linhas.append(' '.join([str(CLASSES.index(nomes[int(p[0])]))] + p[1:5]))
        shutil.copy(img, f'{destino_imgs}/{prefixo}{os.path.basename(img)}')
        with open(f'{destino_lbls}/{prefixo}{base}.txt', 'w') as f:
            f.write('\n'.join(linhas) + ('\n' if linhas else ''))
        n += 1
    return n


FRAMES_NOVOS = []
if TEM_FRAMES_NOVOS:
    !unzip -q -o "{ZIPS}/novos_frames_v7.zip" -d /content/fv7
    n = remapear_roboflow('/content/fv7', f'{DATA_DIR}/train/images', f'{DATA_DIR}/train/labels', 'v7frame_')
    FRAMES_NOVOS = glob.glob(f'{DATA_DIR}/train/images/v7frame_*')
    print('frames novos no treino:', n)

TESTE_NOVO_DIR = '/content/teste_real_v7'
if TEM_TESTE_NOVO:
    for t in ['images', 'labels']:
        os.makedirs(f'{TESTE_NOVO_DIR}/{t}', exist_ok=True)
    !unzip -q -o "{ZIPS}/teste_real_v7.zip" -d /content/tv7
    print('teste real novo:', remapear_roboflow('/content/tv7', f'{TESTE_NOVO_DIR}/images', f'{TESTE_NOVO_DIR}/labels', ''), 'imagens')

In [ ]:
# Lista de treino com reamostragem + checagem antes de gastar GPU
from collections import Counter

for split in ['train', 'val', 'test']:
    imgs = os.listdir(f'{DATA_DIR}/{split}/images')
    lbls = os.listdir(f'{DATA_DIR}/{split}/labels')
    assert imgs and len(imgs) == len(lbls), f'{split}: {len(imgs)} imagens x {len(lbls)} labels'

linhas, caixas, repetidas = [], Counter(), 0
novos = set(FRAMES_NOVOS)
for img in sorted(glob.glob(f'{DATA_DIR}/train/images/*')):
    lbl = f'{DATA_DIR}/train/labels/' + os.path.splitext(os.path.basename(img))[0] + '.txt'
    classes = [CLASSES[int(l.split()[0])] for l in open(lbl) if l.split()]
    caixas.update(classes)
    rep = REPETE_FRAMES_NOVOS if img in novos else (REPETE_LUVA_BOTA if {'gloves', 'boots'} & set(classes) else 1)
    repetidas += rep - 1
    linhas += [img] * rep
with open(f'{DATA_DIR}/train.txt', 'w') as f:
    f.write('\n'.join(linhas) + '\n')
print(f'treino: {len(os.listdir(f"{DATA_DIR}/train/images"))} imagens unicas, {len(linhas)} por epoca (+{repetidas} repeticoes)')
print('caixas no treino:', dict(caixas))
for split in ['val', 'test']:
    print(split, len(os.listdir(f'{DATA_DIR}/{split}/images')), 'imagens')
print('real_world_test:', len(os.listdir(f'{REAL_TEST_DIR}/images')), 'imagens (fora do treino)')

with open(f'{DATA_DIR}/data.yaml', 'w') as f:
    f.write(f'train: {DATA_DIR}/train.txt\nval: {DATA_DIR}/val/images\ntest: {DATA_DIR}/test/images\n')
    f.write(f'nc: 5\nnames: {CLASSES}\n')
for pasta in [REAL_TEST_DIR] + ([TESTE_NOVO_DIR] if TEM_TESTE_NOVO else []):
    with open(f'{pasta}/data.yaml', 'w') as f:
        f.write(f'train: {pasta}/images\nval: {pasta}/images\n')
        f.write(f'nc: 5\nnames: {CLASSES}\n')

## Treino (retoma sozinho se a sessao tiver caido)

Augmentations de video (Albumentations, aplicadas por imagem, so mudam os pixels - as caixas ficam iguais):

| Transformacao | Prob. | Simula |
|---|---|---|
| MotionBlur (3-11 px) | 0.25 | pessoa/camera em movimento |
| ImageCompression (q 30-90) | 0.30 | compressao do video do YouTube |
| Downscale (0.35-0.75) | 0.20 | video em baixa resolucao (480p) |
| GaussNoise | 0.15 | sensor em pouca luz |
| RandomBrightnessContrast | 0.30 | iluminacao de galpao |
| CLAHE / ToGray | 0.05 | contraste e cor variados |

`scale=0.6` (padrao 0.5) aumenta a variacao de tamanho no mosaico, ajudando objetos pequenos. `batch=0.70` = o Ultralytics escolhe o batch para usar ~70% da memoria da GPU.

In [ ]:
%pip install -q -U ultralytics albumentations
import albumentations as A
import torch
from ultralytics import YOLO

AUG_VIDEO = [
    A.MotionBlur(blur_limit=(3, 11), p=0.25),
    A.ImageCompression(quality_range=(30, 90), p=0.30),
    A.Downscale(scale_range=(0.35, 0.75), p=0.20),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.15),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.30),
    A.CLAHE(p=0.05),
    A.ToGray(p=0.05),
]

last = f'{RUN_DIR}/weights/last.pt'
terminado = False
if os.path.exists(last):
    ck = torch.load(last, map_location='cpu', weights_only=False)
    # ao terminar, o Ultralytics remove o otimizador e marca epoch = -1
    terminado = ck.get('epoch', -1) == -1
    print('treino anterior encontrado:', 'TERMINADO' if terminado else f"parou na epoca {ck.get('epoch')}")

if terminado:
    print('Pulando o treino - indo direto para a avaliacao.')
elif os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO(MODELO_BASE).train(
        data=f'{DATA_DIR}/data.yaml',
        epochs=EPOCHS,
        patience=PATIENCE,
        batch=0.70,
        imgsz=IMGSZ,
        scale=0.6,
        close_mosaic=8,
        augmentations=AUG_VIDEO,
        project=RUNS,
        name=RUN_NAME,
        exist_ok=True,
    )

## Veredito: v6 x v7 no conjunto de teste real

Frames de video anotados a mao que nunca entraram em nenhum treino (25 do teste real original e, se existir, o teste real novo). Avaliado em 960 (tamanho do app). Para o TCC, use os numeros do `scripts/avaliar_teste_real.py` no PC (mesmo protocolo de todas as versoes, com intervalo de confianca por bootstrap); esta celula e so o primeiro veredito.

In [ ]:
import pandas as pd


def avaliar(modelo, data, nome, imgsz, conjunto, split='val'):
    r = YOLO(modelo).val(data=data, imgsz=imgsz, split=split, batch=8, conf=0.001, iou=0.6, plots=False,
                         verbose=False, project='/content/val_tmp', name=f'{nome}_{conjunto}', exist_ok=True)
    linhas = [{'conjunto': conjunto, 'modelo': nome, 'classe': 'GERAL', 'mAP50': r.box.map50, 'mAP50-95': r.box.map,
               'precisao': r.box.mp, 'recall': r.box.mr}]
    for i, c in enumerate(r.box.ap_class_index):
        linhas.append({'conjunto': conjunto, 'modelo': nome, 'classe': CLASSES[int(c)], 'mAP50': r.box.ap50[i],
                       'mAP50-95': r.box.ap[i], 'precisao': r.box.p[i], 'recall': r.box.r[i]})
    return linhas


V7_MODEL = f'{RUN_DIR}/weights/best.pt'
conjuntos = [('teste_real_25', REAL_TEST_DIR)] + ([('teste_real_v7', TESTE_NOVO_DIR)] if TEM_TESTE_NOVO else [])
linhas = []
for conjunto, pasta in conjuntos:
    if os.path.exists(V6_MODEL):
        linhas += avaliar(V6_MODEL, f'{pasta}/data.yaml', 'v6', 960, conjunto)
    linhas += avaliar(V7_MODEL, f'{pasta}/data.yaml', 'v7', 960, conjunto)
res = pd.DataFrame(linhas).round(3)
texto = ''
for conjunto, _ in conjuntos:
    t = res[res.conjunto == conjunto].pivot(index='classe', columns='modelo', values=['mAP50', 'recall', 'precisao'])
    texto += f'=== {conjunto} (imgsz 960) ===\n' + t.to_string() + '\n\n'
print(texto)

val = pd.DataFrame(avaliar(V7_MODEL, f'{DATA_DIR}/data.yaml', 'v7', IMGSZ, 'validacao')).round(3)
print('=== v7 na validacao (otimista, mesma distribuicao do treino) ===')
print(val.to_string(index=False))

res.to_csv(f'{RUN_DIR}/comparacao_teste_real.csv', index=False)
with open(f'{RUN_DIR}/comparacao_teste_real.txt', 'w') as f:
    f.write(texto + '=== v7 na validacao ===\n' + val.to_string(index=False) + '\n')
print(f'\nSalvo em {RUN_DIR}/comparacao_teste_real.csv e .txt')